# Load ALL data: Baselines and Concurrent runs

The following script will fetch data from the `SbatchMan` (baselines + concurrent metadata) and `workerpool_out` (concurrent outputs) directories.

In [ ]:
%load_ext autoreload
%autoreload 2

from parse_raw_data import SYSTEM_ORDER, SYSTEMS, build_baselines_dict, compute_slowdowns, parse_baselines, parse_concurrent, get_baselines_dataframe
print(f'{SYSTEMS=}')
print(f'{SYSTEM_ORDER=}')

# Manual override of systems to consider
SYSTEMS = ['lumi']

In [ ]:
baselines = parse_baselines(SYSTEMS)
baselines_dict = build_baselines_dict(baselines)

In [ ]:
concurrent = parse_concurrent(SYSTEMS)
compute_slowdowns(baselines_dict, concurrent)
# Slowdowns are saved directly into `ConcurrentRun` instances

In [ ]:
df = get_baselines_dataframe(baselines_dict)
# df = df[df['system'].isin(['lumi'])]
print(df['system'].unique())
df[['system', 'strategy', 'model', 'gpus', 'comm_relevance', 'comm_relevance_min', 'comm_relevance_max']]

In [ ]:
print('Loaded concurrent runs:')
for s, c in concurrent.items():
    print(f"{s:<20} -> {len(c)}")

# Plots

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter 
from pathlib import Path
import math
from data_types import * #STRATEGY_ORDER, PLACEMENT_ORDER, SYSTEM_ORDER, SYSTEM_NAMES_MAP, Placement, RunKey, ConcurrentRun, Strategy, Model, ensure_model, ensure_placement, ensure_strategy
from collections import defaultdict
import os
from typing import Dict, List, Union, Tuple, Optional, Literal

sns.set_theme(palette="muted", style="white", font_scale=1.5) 
mpl.rcParams.update({
    #"text.usetex": True,
    #"text.latex.preamble": r"\usepackage{siunitx} \usepackage{sansmath} \sansmath",
    "font.size": 16,
    "axes.titlesize": 24,
    "axes.titleweight": "bold",
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 9,
    "legend.title_fontsize": 12,
    "figure.titlesize": 20,
    "axes.spines.right": False, # Disable top and left spines by default (Tufte style)
    "axes.spines.top": False,
})

## Network Relevance

In [ ]:
from statistics import mean


data = []
for baseline, m in baselines_dict.items():
    if not m.comm_relevance:
        print('No relevance for')
        print(baseline)
        continue
    if baseline.gpus == 224:
        continue
    data.append({
        "system":       baseline.system,
        "strategy":     baseline.strategy.short(),
        "model":        baseline.model.short(),
        "placement":    baseline.placement_class.value,
        "gpus":         baseline.gpus,
        "comm_mean":    mean(m.comm_relevance),
        "comm_min":     min(m.comm_relevance),
        "comm_max":     max(m.comm_relevance),
    })
df = pd.DataFrame(data)
df["strategy_model"] = df["strategy"] + "\n" + df["model"]
# df[(df['strategy']=="DP+PP") & (df['model']=="llama3-8b") & (df['placement']=="na")]
df = df[df['comm_mean'] <= 1.0]

strategy_order_labels = [s.short() for s in STRATEGY_ORDER]
placement_order_labels = [p.value for p in PLACEMENT_ORDER]

df["strategy"] = pd.Categorical(
    df["strategy"],
    categories=strategy_order_labels,
    ordered=True,
)
df["placement"] = pd.Categorical(
    df["placement"],
    categories=placement_order_labels, 
    ordered=True,
)
strategy_model_order = [
    f"{s.short()}\n{m}"
    for s in STRATEGY_ORDER
    for m in sorted(df["model"].unique())
]
# df["strategy_model"] = pd.Categorical(
#     df["strategy_model"],
#     categories=strategy_model_order,
#     ordered=True,
# )
df = df.sort_values(
    by=["system", "strategy", "model", "placement", "gpus"]
)
df

In [ ]:
def format_gpus(gpus: int, *_):
    if gpus > 512:
        return f'{int(gpus / 1e3)}K'
    return str(gpus)

def plot_comm_scatter(
    df: pd.DataFrame,
    output_path: Path,
    nrows: int | None = None,
    ncols: int | None = None,
):
    # ----------------------------
    # Encode x/y positions
    # ----------------------------
    # Build ordered strategy_model list respecting STRATEGY_ORDER
    strategy_model_order = [
        f"{s.short()}\n{m}"
        for s in STRATEGY_ORDER
        for m in sorted(df["model"].unique())
    ]
    # Only keep those that actually appear in the data
    strategies = [s for s in strategy_model_order if s in df["strategy_model"].values]

    strategy_to_y = {s: i for i, s in enumerate(strategies)}

    placements = df["placement"].unique()
    placement_to_offset = {
        p: 2.5 for i, p in enumerate(placements)
    }

    df = df.copy()
    df["y_base"] = df["strategy_model"].map(strategy_to_y)
    df["y"] = df["y_base"] + df["placement"].map(placement_to_offset).astype(float)
    df["comm_perc"] = df["comm_mean"] * 100.0

    systems = list(df["system"].unique())
    n_systems = len(systems)

    # ----------------------------
    # Layout (custom rows/cols)
    # ----------------------------
    if nrows is None and ncols is None:
        ncols = math.ceil(math.sqrt(n_systems))
        nrows = math.ceil(n_systems / ncols)
    elif nrows is None:
        nrows = math.ceil(n_systems / ncols)
    elif ncols is None:
        ncols = math.ceil(n_systems / nrows)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(8 * ncols, 6 * nrows),
        sharex=False,
        sharey=True,
    )

    axes = np.array(axes).reshape(-1)

    cmap = plt.cm.viridis

    systems = [s for s in SYSTEM_ORDER if s in systems]

    # ----------------------------
    # Plot
    # ----------------------------
    for ax_i, (ax, system) in enumerate(zip(axes, systems)):
        subdf = df[df["system"] == system]

        # system-specific x mapping (SWAPPED)
        gpus = sorted(subdf["gpus"].unique())
        gpu_to_x = {g: i for i, g in enumerate(gpus)}

        subdf = subdf.copy()
        subdf["x"] = subdf["gpus"].map(gpu_to_x)
        strategy_order_labels = [s.short() for s in STRATEGY_ORDER]
        subdf["strategy"] = pd.Categorical(
            subdf["strategy"],
            categories=strategy_order_labels,
            ordered=True,
        )
        subdf = subdf.sort_values(['strategy', 'model', 'gpus'])

        ax.grid(True, alpha=0.3)

        agg = (
            subdf.groupby(["y_base", "x", "gpus"], sort=True)["comm_perc"]
            .agg(["min", "max", "mean"])
            .reset_index()
        )

        ax.scatter(
            agg["x"],          # GPUs → x-axis
            agg["y_base"],     # strategies → y-axis
            c=agg["max"],
            cmap=cmap,
            vmin=df["comm_perc"].min(),
            vmax=df["comm_perc"].max(),
            marker="s",
            s=1400,
            edgecolor="black",
            linewidth=1.0,
            alpha=0.9,
        )

        for _, row in agg.iterrows():
            if abs(row["min"] - row["max"]) < 1.0:
                label = f"{row['min']:.0f}"
                short_label = True
            else:
                label = f"{row['min']:.0f}-{row['max']:.0f}"
                short_label = False
                
            color="white"
            if row["min"] >= 60:
                color="black"

            ax.annotate(
                label,
                xy=(row["x"], row["y_base"]),
                ha="center",
                va="center",
                fontsize=18 if short_label else 12,
                fontweight="bold",
                color=color,
            )

        ax.set_title(SYSTEM_NAMES_MAP[system])

        # X axis → GPUs (SWAPPED)
        # ax.set_xscale('log', base=2)
        ax.set_xticks(range(len(gpus)))
        ax.set_xticklabels([format_gpus(g) for g in gpus], rotation=35)
        ax.set_xlabel("GPUs")

        # Y axis → strategies (SWAPPED)
        ax.set_yticks(range(len(strategies)))
        if ax_i == 0:
            ax.set_yticklabels(strategies)
            ax.set_ylabel("Strategy")
            
    # Remove unused axes
    for ax in axes[n_systems:]:
        ax.remove()

    # ----------------------------
    # Global colorbar (right, outside)
    # ----------------------------
    sm = plt.cm.ScalarMappable(cmap=cmap)
    sm.set_array(df["comm_perc"])

    fig.subplots_adjust(right=0.89, wspace=0.05)

    cbar_ax = fig.add_axes([0.90, 0.15, 0.01, 0.7])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Communication Time (%)")

    # fig.subplots_adjust(top=0.85)

    # ----------------------------
    # Save
    # ----------------------------
    output_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(Path(output_path).with_suffix('.pdf'), dpi=300, bbox_inches="tight")
    plt.close()


plot_comm_scatter(df, Path('plots/breakdown/comm_relevance'), ncols=6)

## Scaling

In [ ]:
def _placement_linestyles(placements: List[Union[str, Placement]]) -> Dict[str, str]:
    ps = {ensure_placement(p) for p in placements}
    return {str(p): p.linestyle() for p in PLACEMENT_ORDER if p in ps}

def _placement_markers(placements: List[Union[str, Placement]]) -> Dict[str, str]:
    ps = {ensure_placement(p) for p in placements}
    return {str(p): p.marker() for p in PLACEMENT_ORDER if p in ps}

def _strategy_markers(strategies: List[Union[str, Strategy]]) -> Dict[str, str]:
    ss = {ensure_strategy(s) for s in strategies}
    return {str(s): s.marker() for s in STRATEGY_ORDER if s in ss}

def _strategy_colors(strategies: List[Union[str, Strategy]]) -> Dict[str, str]:
    ss = {ensure_strategy(s) for s in strategies}
    return {str(s): s.color() for s in STRATEGY_ORDER if s in ss}

def aggregate_placements(summary: pd.DataFrame, agg_type: str = "geomean") -> pd.DataFrame:
    """
    Collapse all placements of the same
    (strategy, model, system, gpus, gpu_model) into a single row.
    """
    agg = (
        summary
        .groupby(
            ["strategy", "model", "system", "gpus"],
            as_index=False,
        )
        .agg(
            system           =("system",            "first"),
            nodes            =("gpus",              "first"),
            throughput_median=("throughput_median",  agg_type),
            throughput_std   =("throughput_std",     agg_type),
            runtime_mean     =("runtime_mean",       agg_type),
            barrier_mean     =("barrier_mean",       agg_type),
            compute_mean     =("compute_mean",       agg_type),
            comm_pct         =("comm_pct",           agg_type),
            compute_pct      =("compute_pct",        agg_type),
        )
    )
    agg["strategy"]  = agg["strategy"]
    agg["placement"] = ""
    return agg




def _save_or_show(fig, output_file: Optional[str], label: str = "Plot"):
    if output_file:
        Path(output_file).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(Path(output_file).with_suffix('.pdf'), dpi=300, bbox_inches="tight")
        print(f"  {label} saved to: {output_file}")
        plt.close(fig)
    else:
        plt.show()

def _model_linestyles(models: List[Union[str, Model]]) -> Dict[str, str]:
    styles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1))]
    ms = sorted({ensure_model(m) for m in models}, key=lambda x: x.value)
    return {str(m): styles[i % len(styles)] for i, m in enumerate(ms)}

class OverlapHandler:
    """
    Tracks display-space bounding boxes of already-placed annotations
    so subsequent labels can be nudged to a free spot.
    """

    def __init__(self, padding_px: float = 2.0):
        self.padding_px = padding_px
        self._placed: list[tuple[float, float, float, float]] = []

    def _expand(self, box: tuple) -> tuple:
        p = self.padding_px
        x0, y0, x1, y1 = box
        return (x0 - p, y0 - p, x1 + p, y1 + p)

    def overlaps_any(self, box: tuple) -> bool:
        x0, y0, x1, y1 = self._expand(box)
        for bx0, by0, bx1, by1 in self._placed:
            if x0 < bx1 and x1 > bx0 and y0 < by1 and y1 > by0:
                return True
        return False

    def register(self, box: tuple):
        self._placed.append(box)


def _make_label(
    strategy: str,
    model: str,
    placement: str,
    system: str,
    include_system: bool = True,
) -> str:
    """Human-readable series label including model name."""
    place = ensure_placement(placement).display(new_line=False, short=True)
    system_disp = system
    strategy = ensure_strategy(strategy)
    model = ensure_model(model)
    parts = [strategy.short(), model.short()]
    parts.append(place)
    if include_system:
        parts.append(system_disp)
    return " - ".join(parts)


def _dedup_gpus(grp: pd.DataFrame, label: str, metric: str) -> pd.DataFrame:
    """Keep the highest-throughput row when the same GPU count appears twice."""
    if grp["gpus"].duplicated().any():
        dup_counts = grp["gpus"].value_counts()
        for gpu, count in dup_counts[dup_counts > 1].items():
            best = grp[grp["gpus"] == gpu][f"throughput_{metric}"].max()
            print(f"[WARN] Duplicate GPU entries for {label}: gpus={gpu} "
                  f"({count} rows) → keeping max {best:.3f}")
        grp = grp.loc[grp.groupby("gpus")[f"throughput_{metric}"].idxmax()]
    return grp.sort_values("gpus")

def _label_text(y_val: float, plot_type: str) -> str:
    """Format the annotation string for a single point."""
    if plot_type == "efficiency":
        return f"{y_val * 100:.0f}%"
    v = float(y_val)
    if v < 100:
        return str(int(v))
    if v < 1_000:
        return f"{v / 1_000:.1f}K"
    if v < 1_000_000:
        return f"{int(v / 1_000)}K"
    if v < 1_000_000_000:
        return f"{int(v / 1_000_000)}M"
    return f"{int(v / 1_000_000_000)}B"

STRATEGY_LABEL_OFFSETS: dict[Strategy, int] = {
    Strategy.DP:           2,  
    Strategy.FSDP:         40,   
    Strategy.DP_PP:       -1,   
    Strategy.DP_PP_TP:     2,   
    Strategy.DP_PP_EXPERT:-4,   
}

def annotate_points(
    ax: plt.Axes,
    x_vals,
    y_vals,
    *,
    label: str,
    color: str,
    plot_type: str,
    strategy: Optional[str] = None,
    label_filters: Optional[List[str]],
    label_gpus: Optional[List[int]],
    label_every_nth: int,
    series_index: int,
    fontsize: int,
    overlap_handler: OverlapHandler,
    renderer,
) -> None:
    x_arr = np.asarray(x_vals, dtype=float)
    y_arr = np.asarray(y_vals, dtype=float)

    # ── series-level filter ───────────────────────────────────────────────
    if label_filters is not None:
        if not label_filters:
            return
        if not any(re.search(pat, label) for pat in label_filters):
            return

    # ── determine starting offset for this strategy ───────────────────────
    try:
        start_off = STRATEGY_LABEL_OFFSETS[Strategy(strategy)] if strategy else 0
    except ValueError:
        start_off = 0   # unknown strategy → fall back to on-marker

    MARKER_PX = 5
    MAX_R     = 10
    STEP      =  6

    def _offsets(start: int):
        """
        Yield offsets beginning at `start`, then alternating outward.
        If start > 0 we prefer going further up first; if start < 0, further down.
        If start == 0 we alternate symmetrically.
        """
        yield start
        sign = 1 if start >= 0 else -1
        for step in range(STEP, MAX_R + 1, STEP):
            yield start + sign * step          # preferred direction first
            yield start - sign * step          # opposite direction second

    # ── iterate over points ───────────────────────────────────────────────
    for pt_idx, (xv, yv) in enumerate(zip(x_arr, y_arr)):

        if label_gpus is not None and int(xv) not in label_gpus:
            continue

        if (pt_idx + series_index) % max(1, label_every_nth) != 0:
            continue

        val_str = _label_text(yv, plot_type)

        # Register the marker dot as occupied
        disp_pt = ax.transData.transform((xv, yv))
        overlap_handler.register((
            disp_pt[0] - MARKER_PX, disp_pt[1] - MARKER_PX,
            disp_pt[0] + MARKER_PX, disp_pt[1] + MARKER_PX,
        ))

        placed = False

        for y_off in _offsets(start_off):
            arrow = None if y_off == 0 else dict(
                arrowstyle="-", color=color, alpha=0.4, lw=0.8
            )
            va = "center" if y_off == 0 else ("bottom" if y_off > 0 else "top")

            ann = ax.annotate(
                val_str,
                xy=(xv, yv),
                xytext=(0, y_off),
                textcoords="offset points",
                fontsize=fontsize,
                color=color,
                ha="center",
                va=va,
                arrowprops=arrow,
                bbox=dict(
                    boxstyle="round,pad=0.15",
                    fc="white",
                    ec=color,
                    alpha=0.7,
                    lw=0.6,
                ),
                annotation_clip=False,
            )

            try:
                bbox = ann.get_window_extent(renderer=renderer)
            except Exception:
                ann.remove()
                continue

            box = (bbox.x0, bbox.y0, bbox.x1, bbox.y1)

            if not overlap_handler.overlaps_any(box):
                overlap_handler.register(box)
                placed = True
                break
            else:
                ann.remove()

        # fall-back
        if not placed:
            fallback_off = start_off + (MAX_R if start_off >= 0 else -MAX_R)
            ann = ax.annotate(
                val_str,
                xy=(xv, yv),
                xytext=(0, fallback_off),
                textcoords="offset points",
                fontsize=fontsize,
                color=color,
                ha="center",
                va="bottom" if fallback_off >= 0 else "top",
                arrowprops=dict(arrowstyle="-", color=color, alpha=0.4, lw=0.8),
                bbox=dict(
                    boxstyle="round,pad=0.15",
                    fc="white",
                    ec=color,
                    alpha=0.7,
                    lw=0.6,
                ),
                annotation_clip=False,
            )
            try:
                bbox = ann.get_window_extent(renderer=renderer)
                overlap_handler.register((bbox.x0, bbox.y0, bbox.x1, bbox.y1))
            except Exception:
                pass
        

In [ ]:
def plot_global_faceted(
    combined: pd.DataFrame,
    output_dir: Path,
    metric: str,
    pfx: str = "",
    no_ideal: bool = False,
    aggregate_placements_flag: bool = False,
    n_rows: int = 1,
    plot_type: str = "scaling",   # "scaling" | "efficiency"
    figsize_per_cell: Tuple[int, int] = (8, 6),
    label_filters: Optional[List[str]] = None,
    label_gpus: Optional[List[int]] = None,
    label_every_nth: int = 1,
    label_fontsize: int = 12,
):
    """
    One subplot per system, laid out in a grid of *n_rows* rows.

    Parameters
    ----------
    combined : pd.DataFrame
        Concatenated summary from all systems.
    output_dir : Path
        Directory where the figure is saved.
    metric : str
        Throughput column suffix (e.g. "mean", "median").
    pfx : str
        Filename prefix.
    no_ideal : bool
        Suppress ideal-scaling reference lines.
    aggregate_placements_flag : bool
        Average across placements before plotting each panel.
    n_rows : int
        Number of rows in the subplot grid.
    plot_type : str
        "scaling" or "efficiency".
    figsize_per_cell : (width, height)
        Size of each individual subplot cell in inches.
    label_filters : list[str] | None
        Substrings / regexes that select which series get point labels.
        None → label all series.  Pass an empty list [] to suppress all labels.
    label_gpus : list[int] | None
        Annotate only these GPU counts.  None → respect *label_every_nth*.
    label_every_nth : int
        Annotate every n-th point along each series (1 = all points).
    label_fontsize : int
        Font size used for point annotations.
    """
    import math
    from matplotlib.ticker import FuncFormatter

    sys_order = SYSTEM_ORDER
    systems = [s for s in sys_order if s in combined["system"].unique()]
    n_sys   = len(systems)
    if n_sys == 0:
        print("  [SKIP] No systems in combined summary.")
        return

    n_rows  = max(1, min(n_rows, n_sys))
    n_cols  = math.ceil(n_sys / n_rows)
    mode_tag = "aggr" if aggregate_placements_flag else "all"

    fig_w = n_cols * figsize_per_cell[0]
    fig_h = n_rows * figsize_per_cell[1]
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)

    axes_flat = [axes[r][c] for r in range(n_rows) for c in range(n_cols)]
    for ax in axes_flat[n_sys:]:
        ax.set_visible(False)

    global_handles = []
    global_labels  = []

    def format_throughput(throughput: float, *_):
        if throughput < 100:
            return str(int(throughput))
        if throughput < 1e3:
            return f"{(throughput / 1e3):.1f}K"
        if throughput < 1e6:
            return f"{int(throughput / 1e3)}K"
        if throughput < 1e9:
            return f"{int(throughput / 1e6)}M"
        return f"{int(throughput / 1e9)}B"

    def format_gpus(gpus: int, *_):
        if gpus == 224:
            return ''
        if gpus > 512:
            return f'{int(gpus / 1e3)}K'
        return str(gpus)

    def format_efficiency(e: float, *_):
        return f'{int(e * 100.0)}'

    for idx, system_name in enumerate(systems):
        ax  = axes_flat[idx]
        sub = combined[combined["system"] == system_name].copy()

        plot_summary = aggregate_placements(sub) if aggregate_placements_flag else sub

        gpus_local    = sorted(plot_summary["gpus"].unique())
        place_ls      = _placement_linestyles(plot_summary["placement"].unique())
        model_ls      = _model_linestyles(plot_summary["model"].unique())
        place_markers = _placement_markers(plot_summary["placement"].unique())
        base_color    = _strategy_colors(plot_summary["strategy"].unique())

        min_eff    = 1.0
        
        # ── draw once so transforms/log-scale ticks are initialized ──────────
        # This is the single draw() call for the entire subplot. It is O(1)
        # per subplot, not O(n_points), so performance stays acceptable.
        fig.set_layout_engine("none")          # prevents tight_layout from firing early
        renderer = fig.canvas.get_renderer() if hasattr(fig.canvas, "get_renderer") \
            else plt.matplotlib.backends.backend_agg.FigureCanvasAgg(fig).get_renderer()
        
        handler = OverlapHandler(padding_px=1.0)
        series_idx = 0
        
        for (strategy, system, model, placement), grp in plot_summary.groupby(
            ["strategy", "system", "model", "placement"]
        ):
            color  = base_color[strategy]
            ls     = model_ls[model]
            marker = place_markers.get(placement, "o")
            label  = _make_label(strategy, model, placement, system, include_system=False)

            grp = _dedup_gpus(grp, label, metric)

            if plot_type == "scaling":
                ax.errorbar(
                    grp["gpus"], grp[f"throughput_{metric}"],
                    yerr=grp["throughput_std"],
                    label=label, color=color, marker=marker, linestyle=ls,
                    linewidth=2, markersize=10, capsize=4,
                )
                if not (system == 'jupiter' and ( \
                    (ensure_strategy(strategy) == Strategy.DP and ensure_placement(placement) != Placement.INTRA_L1_RANDOM) \
                    or (ensure_strategy(strategy) == Strategy.DP_PP_TP and ensure_placement(placement) != Placement.INTER_GROUP_RANDOM) \
                    or (ensure_strategy(strategy) == Strategy.DP_PP_EXPERT and ensure_placement(placement) != Placement.INTRA_GROUP_RANDOM) \
                    )):
                    annotate_points(
                        ax, grp["gpus"], grp[f"throughput_{metric}"],
                        label=label, color=color, plot_type="scaling", strategy=ensure_strategy(strategy),
                        label_filters=label_filters, label_gpus=label_gpus,
                        label_every_nth=label_every_nth, series_index=series_idx,
                        fontsize=label_fontsize, overlap_handler=handler, renderer=renderer,
                    )
                if not no_ideal:
                    base_row   = grp.iloc[0]
                    gpus_range = np.array([g for g in grp["gpus"].unique()
                                           if g >= base_row["gpus"]])
                    ideal = base_row[f"throughput_{metric}"] * (gpus_range / base_row["gpus"])
                    ax.plot(gpus_range, ideal, color=color,
                            linestyle=":", linewidth=2, alpha=0.35)

            else:  # efficiency
                g0  = grp.iloc[0]["gpus"]
                T0  = grp.iloc[0][f"throughput_{metric}"]

                if (
                    system_name == 'lumi'
                    and strategy in [Strategy.DP, Strategy.DP_PP, Strategy.FSDP]
                    # and (grp["gpus"] >= 8).any()
                ):
                    row_8 = grp[grp["gpus"] == 8]
                    if not row_8.empty:
                        print(f'LUMI efficiency base {grp.iloc[0][f"throughput_{metric}"]} ---> {row_8.iloc[0][f"throughput_{metric}"]}')
                        g0 = row_8.iloc[0]["gpus"]
                        T0 = row_8.iloc[0][f"throughput_{metric}"]
                        
                eff = (grp[f"throughput_{metric}"] / T0) * (g0 / grp["gpus"])
                min_eff = min(min_eff, float(eff.min()))
                ax.plot(grp["gpus"], eff, label=label,
                        color=color, marker=marker, linestyle=ls,
                        linewidth=2, markersize=10)
                # annotate_points(
                #     ax, grp["gpus"], eff.values,
                #     label=label, color=color, plot_type="efficiency",
                #     label_filters=label_filters, label_gpus=label_gpus,
                #     label_every_nth=label_every_nth, series_index=series_idx,
                #     fontsize=label_fontsize, overlap_handler=handler, renderer=renderer,
                # )

            series_idx += 1

        # Axis formatting
        if plot_type == "scaling":
            ax.set_xscale("log", base=2)
            ax.set_yscale("log", base=2)
            if idx % n_cols == 0:
                ax.set_ylabel("Throughput (samples/s)", fontsize=22)
            if not no_ideal:
                ax.plot([], [], color="gray", linestyle=":", linewidth=2,
                        alpha=0.9, label="Ideal")
            ax.yaxis.set_major_formatter(FuncFormatter(format_throughput))
            ax.yaxis.set_tick_params(labelsize=20)
        else:  # efficiency
            ax.axhline(1.0, linestyle=":", linewidth=2, alpha=0.9,
                       color="gray", label="Ideal")
            ax.set_xscale("log", base=2)
            if idx % n_cols == 0:
                ax.set_ylabel("Parallel Efficiency (%)", fontsize=22)
            ax.yaxis.set_major_formatter(FuncFormatter(format_efficiency))
            ax.yaxis.set_tick_params(labelsize=20)

        ax.set_xticks(gpus_local)
        ax.xaxis.set_major_formatter(FuncFormatter(format_gpus))
        ax.xaxis.set_tick_params(
            labelsize=19 if system_name != 'lumi' else 16,
        )
        ax.set_title(SYSTEM_NAMES_MAP.get(system_name, system_name),
                     fontsize=24, fontweight="bold")
        if plot_type != "scaling":
            ax.set_xlabel("GPUs", fontsize=20)
        ax.grid(True, alpha=0.45)

        handles, labels = ax.get_legend_handles_labels()
        for h, l in zip(handles, labels):
            if l not in global_labels:
                global_handles.append(h)
                global_labels.append(l)

    # One global legend at the bottom
    if global_labels and plot_type == "efficiency":
        legend_properties = {'weight': 'bold', 'size': 14}
        sorted_pairs = sorted(zip(global_handles, global_labels),
                              key=lambda x: x[1].split('-'))
        global_handles, global_labels = zip(*sorted_pairs)
        fig.legend(
            list(global_handles),
            list(global_labels),
            loc="lower center",
            ncol=min(9, max(1, len(global_labels))),
            fontsize=20,
            frameon=False,
            prop=legend_properties,
        )

    fig.tight_layout(rect=[0, 0.18, 1, 1.0])

    out = str(output_dir / f"{pfx}global_{mode_tag}_{metric}_{plot_type}_faceted.png")
    _save_or_show(fig, out, f"Faceted {plot_type} plot")

In [ ]:
baselines_df = get_baselines_dataframe(baselines_dict)
baselines_df.map(lambda x: f"{x:.1f}" if isinstance(x, float) else x)
print(baselines_df['system'].unique())

In [ ]:
if baselines_df is not None:
    for metric in ['geomean']: #  ['max', 'median', 'geomean']:
        for plot_type in ["scaling", "efficiency"]:
            plot_global_faceted(
                baselines_df,
                output_dir                = Path('plots/scaling'),
                metric                    = metric,
                no_ideal                  = False,
                aggregate_placements_flag = False,
                n_rows                    = 1,
                plot_type                 = plot_type,
                figsize_per_cell          = (6, 7),
                # label_filters= ['2', '60', '21']
            )
else:
    print("No dataframe!!!")

## Slowdowns Analysis

In [ ]:
def format_gpus(gpus):
    """Return a concise GPU label, e.g. 1024 -> '1K GPU'."""
    if gpus >= 1024 and gpus % 1024 == 0:
        return f"{gpus // 1024}K GPU"
    if gpus >= 1000:
        return f"{gpus / 1024:.1f}K GPU"
    return f"{gpus} GPU"

def _sort_run(run: RunKey):
    """
    Sort key for (strategy, gpus, placement, model_name) tuples.
    """
    s, g, p, m = run.strategy, run.gpus, run.placement_class, run.model
    si = STRATEGY_ORDER.index(s) if s in STRATEGY_ORDER else 99
    pi = PLACEMENT_ORDER.index(p) if p in PLACEMENT_ORDER else 99
    return (pi, si, g, m)

In [ ]:
for system, sys_concurrent in concurrent.items():
    print(f'System: {system}')
    for c in sys_concurrent:
        if system == 'leonardo':
            print(c.job_id)
        if c.job_id == 38409415:
            slowdowns: Dict[RunKey, List[float]] = defaultdict(list)
            for key, measurements in c.multi_runs.items():
                print(key)
                print(baselines_dict[key])
                print(measurements[0]._df)
                print(measurements[0].get_throughput())
                print()
                
            for key, s in c.slowdowns.items():
                slowdowns[key].extend(s)
                
            # print(c)
            print(c.display(include_runs=False))
            print(f"{'Category':55s} {'Runs':>5s}  {'Mean':>6s}  {'Median':>6s}  {'Max':>6s}  {'%>1':>5s}")
            print("-" * 93)
            for cat in sorted(slowdowns, key=_sort_run):
                v = np.array(slowdowns[cat])
                lbl = f"{cat.strategy.short()} / {format_gpus(cat.gpus)} / {cat.placement_class} / {cat.model}"
                print(
                    f"{lbl:55s} {len(v):5d}  {v.mean():6.3f}  {np.median(v):6.3f}  {v.max():6.3f}  "
                    f"{100*np.mean(v>1.05):5.1f}%"
                )
            print()

In [ ]:
from statistics import geometric_mean


def plot_slowdown_heatmaps(
    concurrent: dict[str, List[ConcurrentRun]],
    output_dir: Path,
    metric: Union[Literal['min'], Literal['max'], Literal['median'], Literal['mean'], Literal['geomean']],
    cap: Union[int, None] = None
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    
    def sort_y(y):
        id, in_reservation, system, gpus, pattern, strategies, placement_score = y
        return (
            in_reservation,
            gpus,
            len(strategies),
            placement_score,
        )
    
    agg_metric = np.median
    if metric == 'min':
        agg_metric = np.min
    elif metric == 'max':
        agg_metric = np.max
    elif metric == 'mean':
        agg_metric = np.mean
    elif metric == 'geomean':
        agg_metric = geometric_mean
        
    def agg_fn(slowdowns_stats: SlowdownStats):
        slowdowns = slowdowns_stats.all_ratios()
        # print(f'before cap: {len(slowdowns)}')
        if cap:
            slowdowns_under_cap = slowdowns[(slowdowns-1.0)*100.0 <= cap]
            if slowdowns_under_cap.shape[0] <= 0:
                print('WARNING: cap filtered out all values, rolling back...')
            else:
                slowdowns = slowdowns_under_cap
        # print(f'after cap: {len(slowdowns)}')
        # print()
        return agg_metric(slowdowns)

    records = []

    # -----------------------------
    # 1. Flatten data
    # -----------------------------
    for system, sys_concurrent in concurrent.items():
        print(f'Plotting system {system}')
        for c in sys_concurrent:
            print('\n  ' + c.display())
            strategies = tuple(sorted(s.short() if hasattr(s, "short") else str(s)
                          for s in c.get_distinct_strategies()))
            y_key = (
                c.job_id,
                c.is_in_reservation(),
                c.system,
                c.gpus,
                c._summarize_pattern(),
                strategies,
                c.get_placement_score(),
            )

            for rk, slowdowns in c.slowdowns.items():
                if not slowdowns:
                    continue

                records.append({
                    "system": system,
                    "y": y_key,
                    "strategy": rk.strategy.short(),
                    "model": str(rk.model),
                    "gpus": rk.gpus,
                    "x": (rk.strategy.short(), rk.model.short(), rk.gpus),
                    f"{metric}_slowdown": (float(agg_fn(slowdowns))-1.0)*100.0,
                })

    if not records:
        print("No data available for heatmaps.")
        return

    df = pd.DataFrame(records)

    # -----------------------------
    # 2. Plot per system
    # -----------------------------
    for system in sorted(df["system"].unique()):
        sub = df[df["system"] == system]

        if sub.empty:
            continue

        # -----------------------------
        # 3. Pivot
        # -----------------------------
        pivot = sub.pivot_table(
            index="y",
            columns="x",
            values=f"{metric}_slowdown",
            aggfunc="max",
        )

        if pivot.empty:
            continue

        # pivot = pivot.sort_index(axis=0)
        pivot = pivot.sort_index(axis=1)
        pivot = pivot.sort_index(
            axis=0,
            key=lambda idx: [sort_y(y) for y in idx]
        )

        # -----------------------------
        # 4. Pretty labels
        # -----------------------------
        def fmt_y(y):
            id, in_reservation, sys, gpus, pattern, strategies, placement_score = y
            strat = ','.join(strategies)
            placement_score = f'|  ps={placement_score:.2f}' if placement_score else '' 
            return f"{id} {'(RESV)' if in_reservation else ''} | {gpus}GPU | {pattern} | {strat}{placement_score}"

        pivot.index = [fmt_y(y) for y in pivot.index]
        pivot.columns = [f"{s}\n{m}\n{g}g" for (s, m, g) in pivot.columns]

        # -----------------------------
        # 5. Plot
        # -----------------------------
        plt.figure(figsize=(30, max(12, len(pivot) * 0.45)))

        sns.heatmap(
            pivot,
            annot=True,
            fmt=".0f",
            cmap="viridis",
            linewidths=0.5,
            linecolor="lightgray",
            cbar_kws={"label": f"{metric.capitalize()} Slowdown %"},
        )

        plt.title(f"{metric.capitalize()} Slowdown Heatmap — {system}")
        plt.xlabel("Strategy / Model / GPUs")
        plt.ylabel("Concurrent configuration")
        plt.tight_layout()
        plt.savefig(output_dir / f"{system}_{metric}{'_cap' + str(cap) if cap else ''}_heatmap.png", dpi=300)
        plt.close()
        
for metric in ['max', 'median', 'mean', 'geomean']:
    plot_slowdown_heatmaps(concurrent, Path('plots/heatmaps'), metric, 400)

## Global Summary Slowdown Plots

In [ ]:
def _placement_groups(categories: List[RunKey]):
    """
    Identify contiguous runs of bars that share the same placement.

    Since categories are sorted placement-first by _sort_run, consecutive
    entries with the same placement naturally form a group.

    Returns
    -------
    list of (placement_name, start_idx, end_idx)
    """
    if not categories:
        return []
    groups = []
    current = categories[0].placement_class
    start = 0
    for i, cat in enumerate(categories):
        if cat.placement_class != current:
            groups.append((current, start, i - 1))
            current = cat.placement_class
            start = i
    groups.append((current, start, len(categories) - 1))
    return groups

def _find_clusters_kde(
    arr: np.ndarray,
    cluster_width: float = 0.01,
    min_cluster_frac: float = 0.05,   # NEW: merge clusters smaller than this fraction
    n_grid: int = 512,
) -> list[tuple[float, float, np.ndarray]]:
    """
    KDE-based cluster detection with post-merge of small/overlapping clusters.

    1. Estimate KDE, find local maxima, merge peaks closer than `cluster_width`.
    2. Assign each raw point to its nearest surviving peak.
    3. Post-merge pass: repeatedly merge adjacent clusters if their y-brackets
       overlap OR if either cluster contains fewer than `min_cluster_frac` of
       the total points. Repeats until stable.
    """
    from scipy.signal import find_peaks
    from scipy.stats import gaussian_kde

    if len(arr) < 2:
        return [(arr.min(), arr.max(), arr)]

    kde   = gaussian_kde(arr)
    grid  = np.linspace(arr.min(), arr.max(), n_grid)
    density = kde(grid)

    peak_idx, _ = find_peaks(density)
    if len(peak_idx) == 0:
        peak_idx = np.array([np.argmax(density)])

    # Merge KDE peaks closer than cluster_width
    merged: list[float] = []
    for p in sorted(grid[peak_idx]):
        if merged and (p - merged[-1]) < cluster_width:
            merged[-1] = (merged[-1] + p) / 2
        else:
            merged.append(p)
    merged_peaks = np.array(merged)

    # Assign raw points to nearest peak
    assignments = np.argmin(
        np.abs(arr[:, None] - merged_peaks[None, :]), axis=1
    )
    clusters: list[tuple[float, float, np.ndarray]] = []
    for ci in range(len(merged_peaks)):
        members = arr[assignments == ci]
        if len(members):
            clusters.append((members.min(), members.max(), members))

    # ── Post-merge: collapse overlapping or too-small adjacent clusters ───────
    min_count = max(1, int(min_cluster_frac * len(arr)))
    changed = True
    while changed and len(clusters) > 1:
        changed = False
        merged_clusters: list[tuple[float, float, np.ndarray]] = []
        i = 0
        while i < len(clusters):
            if i + 1 < len(clusters):
                lo_a, hi_a, mem_a = clusters[i]
                lo_b, hi_b, mem_b = clusters[i + 1]
                overlaps  = hi_a >= lo_b          # brackets touch or overlap
                too_small = len(mem_a) < min_count or len(mem_b) < min_count
                if overlaps or too_small:
                    combined = np.concatenate([mem_a, mem_b])
                    merged_clusters.append((combined.min(), combined.max(), combined))
                    i += 2
                    changed = True
                    continue
            merged_clusters.append(clusters[i])
            i += 1
        clusters = merged_clusters

    return clusters

Y_CLIP = 2.0
def _annotate_clipped(ax, categories, slowdowns, y_clip=Y_CLIP):
    """
    Annotate bars/violins/points that exceed y_clip with percentage and max.

    Labels are placed just above the top of the axes frame.
    """
    for i, cat in enumerate(categories):
        vals = np.array(slowdowns[cat])
        above = vals[vals > y_clip]
        if len(above) > 0:
            pct = 100.0 * len(above) / len(vals)
            max_val = float(above.max()) / 100.0
            pct_str = f"{pct:.2f}%" if pct < 1 else f"{pct:.0f}%"
            ax.text(
                i, y_clip * 1.005, f"{pct_str}\n({max_val:.1f}x)",
                ha="center", va="bottom", fontsize=11, color="#d62728",
                fontweight="bold", zorder=5, clip_on=False,
            )

def _setup_grouped_xaxis(ax, categories: List[RunKey], groups, system, y_offset=-0.36):
    """
    Create a two-level x-axis layout.

    Level 1 (tick labels):  short per-bar label — strategy, model, GPU count.
    Level 2 (group labels): centred placement name below each group, with a
                            thin horizontal bracket.

    Vertical dashed lines separate adjacent placement groups.
    """
    x = np.arange(len(categories))

    labels = []
    for cat in categories:
        gpus = cat.gpus
        display_strategy = cat.strategy.short()
        model_label = cat.model.short()
        labels.append(f"{display_strategy}\n{model_label}\n{format_gpus(gpus)}")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=11, ha="center")

    trans = ax.get_xaxis_transform()   # x=data, y=axes fraction

    for placement, start, end in groups:
        if placement == "na":
            continue
        center = (start + end) / 2.0
        display_name = placement.display()

        bracket_y = y_offset + 0.01
        ax.plot(
            [start - 0.15, end + 0.15], [bracket_y, bracket_y],
            transform=trans, color="black", lw=0.7, clip_on=False,
        )
        ax.text(
            center, y_offset, display_name, transform=trans,
            ha="center", va="top", fontsize=11, fontweight="bold",
            clip_on=False,
        )

    for i in range(len(groups) - 1):
        boundary_x = groups[i][2] + 0.5
        ax.axvline(boundary_x, color="grey", ls="--", lw=0.6, alpha=0.6)

    ax.set_xlabel("")

def plot_slowdown_violinplot(
    slowdowns: Dict[RunKey, np.ndarray],
    system,
    output_path,
    y_clip: float = 1.2,
    threshold: float = 1.025,
    cluster_width: float = 0.01,
    min_cluster_frac: float = 0.05,
):
    """
    Violin plot per category.

    - Categories where NO value exceeds `threshold` are excluded.
    - Shows IQR box, median (black line), mean (white diamond), and whiskers
      at 1.5 × IQR, consistent with the original boxplot logic.
    - KDE-based cluster detection: peaks in the density estimate are found,
      then any two peaks closer than `cluster_width` are merged into one.
      Each surviving cluster gets a right-side bracket spanning the min–max
      of its assigned raw points, labelled "26.7% (8/30)".
    """
    categories = sorted(
        [k for k, v in slowdowns.items() if len(v) > 0 and np.any(v > threshold)],
        key=_sort_run,
    )
    if not categories:
        print("No data to plot (violinplot): all categories below threshold.")
        return

    groups  = _placement_groups(categories)
    data    = [slowdowns[cat] for cat in categories]
    strats  = [cat.strategy for cat in categories]
    colors  = [ensure_strategy(s).color() for s in strats]

    x       = np.arange(len(categories))
    fig_w   = max(11, len(categories) * 1.0)
    fig, ax = plt.subplots(figsize=(fig_w, 5))

    # ── Violins ───────────────────────────────────────────────────────────────
    vp = ax.violinplot(
        data, positions=x, widths=0.45,
        showmedians=False, showextrema=False, showmeans=False,
    )
    for i, body in enumerate(vp["bodies"]):
        body.set_facecolor(colors[i])
        body.set_edgecolor("black")
        body.set_linewidth(0.6)
        body.set_alpha(0.75)

    # ── Per-category overlays ─────────────────────────────────────────────────
    BOX_W       = 0.06    # half-width of the IQR box
    BRACKET_X   = 0.28    # x offset of cluster bracket from violin centre
    BRACKET_CAP = 0.04    # half-width of bracket end-caps
    LABEL_PAD   = 0.015   # gap between bracket right edge and label

    for i, arr in enumerate(data):
        n_total          = len(arr)
        q1, med, q3      = np.percentile(arr, [25, 50, 75])
        iqr              = q3 - q1
        lo_whisk         = max(arr.min(), q1 - 1.5 * iqr)
        hi_whisk         = min(arr.max(), q3 + 1.5 * iqr)

        # IQR box
        ax.add_patch(plt.Rectangle(
            (x[i] - BOX_W, q1), 2 * BOX_W, iqr,
            linewidth=0.8, edgecolor="black",
            facecolor="white", alpha=0.6, zorder=3,
        ))

        # Whisker stems + caps
        for y0, y1 in [(lo_whisk, q1), (q3, hi_whisk)]:
            ax.plot([x[i], x[i]], [y0, y1], color="black", lw=0.8, zorder=3)
        for y_cap in (lo_whisk, hi_whisk):
            ax.plot([x[i] - BOX_W, x[i] + BOX_W], [y_cap, y_cap],
                    color="black", lw=0.8, zorder=3)

        # Median line
        ax.plot([x[i] - BOX_W, x[i] + BOX_W], [med, med],
                color="black", lw=1.2, zorder=4)

        # Mean diamond
        ax.plot(x[i], arr.mean(), marker="D", color="white",
                markeredgecolor="black", markersize=4, zorder=5)

        # ── KDE cluster brackets ──────────────────────────────────────────────
        clusters = _find_clusters_kde(arr, cluster_width=cluster_width, min_cluster_frac=min_cluster_frac)
        bx       = x[i] + BRACKET_X
        
        def fmt_nmembers(n: int) -> str:
            if n > 1e3:
                return f'{round(n/1e3)}k'
            return str(n)

        for lo, hi, members in clusters:
            # Clamp to visible range
            vis_lo = max(lo, 0.95)
            vis_hi = min(hi, y_clip)
            if vis_hi <= vis_lo:
                continue

            pct   = 100.0 * len(members) / n_total
            # label = f"{pct:.1f}%\n{fmt_nmembers(len(members))}/{fmt_nmembers(n_total)}"
            if len(members) == n_total:
                label = f"{fmt_nmembers(n_total)}"
            else:
                label = f"{fmt_nmembers(len(members))}/{fmt_nmembers(n_total)}"

            # Vertical bracket line — gray dashed
            ax.plot([bx, bx], [vis_lo, vis_hi],
                    color="gray", lw=0.8, ls="--", zorder=4,
                    solid_capstyle="butt")

            # End caps — gray solid (dashed caps look noisy at small scale)
            for y_end in (vis_lo, vis_hi):
                ax.plot([bx - BRACKET_CAP, bx + BRACKET_CAP], [y_end, y_end],
                        color="gray", lw=1.0, zorder=4)
            # End caps
            for y_end in (vis_lo, vis_hi):
                ax.plot([bx - BRACKET_CAP, bx + BRACKET_CAP], [y_end, y_end],
                        color="black", lw=1.0, zorder=4)

            # Label centred on the bracket
            ax.text(
                bx + BRACKET_CAP + LABEL_PAD, (vis_lo + vis_hi) / 2, label,
                fontsize=10, va="center", ha="left", zorder=6,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7),
            )

    # ── Decorations ───────────────────────────────────────────────────────────
    ax.axhline(1.0, color="black", ls="--", lw=1, alpha=0.8)
    ax.set_ylim(-10, y_clip)
    _annotate_clipped(ax, categories, slowdowns, y_clip=y_clip)
    _setup_grouped_xaxis(ax, categories, groups, system)
    ax.set_ylabel("Slowdown $\\sigma$", fontsize=20)
    ax.grid(True, alpha=0.5)
    def format_slowdown(s: float, *_):
        def fmt(x, suffix):
            return f"{int(x)}{suffix}" if x.is_integer() else f"{x:.1f}{suffix}"
        if s >= 100.0:
            return fmt(s / 100.0, "x")
        else:
            return fmt(s, "%")
    ax.yaxis.set_major_formatter(FuncFormatter(format_slowdown))
    ax.yaxis.set_tick_params(labelsize=18)

    fig.tight_layout()
    fig.savefig(Path(output_path).with_suffix('.pdf'), dpi=300, bbox_inches="tight")
    print(f"Saved -> {output_path}")

In [ ]:
from plot_slowdown import plot_slowdown_violinplot

# CAP = 1000 # 10x
CAP = None

for system, sys_concurrent in concurrent.items():
    # if system != 'leonardo':
    #     continue
    slowdowns_lists: Dict[RunKey, List[float]] = defaultdict(list)
    for c in sys_concurrent:
        for key, s in c.slowdowns.items():
            slowdowns_lists[key].extend(s.all_ratios())
            
    slowdowns: Dict[RunKey, np.ndarray] = {}
    cap_ignored = False
    for k, arr in slowdowns_lists.items():
        len_before = len(arr)
        arr = (np.array(arr)-1.0)*100.0 
        # if CAP or system == 'intel':
        #     cap = 200.0 if system == 'intel' else CAP
        #     slowdowns_under_cap = arr[arr <= cap]
        #     arr = slowdowns_under_cap
        n_excluded = len_before - len(arr)
        if n_excluded > 0:
            print(f'CAP excluded {n_excluded} values for {k.display()}')
        slowdowns[k] = arr
            
    print(f'System: {system}')
    print(f"{'Run Type':50s} {'Runs':>5s}  {'Mean':>8s}  {'Median':>8s}  {'%>5':>5s}")
    print("-" * 85)
    for cat in sorted(slowdowns, key=_sort_run):
        v = slowdowns[cat]
        lbl = f"{cat.strategy.short()} / {format_gpus(cat.gpus)} / {cat.placement_class} / {cat.model}"
        print(
            f"{lbl:50s} {len(v):5d}  {v.mean():8.3f}  {np.median(v):8.3f}  "
            f"{100*np.mean(v>5):5.1f}%"
        )

    output_dir = 'plots/slowdown_all'
    os.makedirs(output_dir, exist_ok=True)
    base_path = os.path.join(output_dir, f"slowdown_{system}.png")

    # plot_slowdown_violin(slowdowns, system, base_path.replace(".png", "_violin.png"))

    # Determine a y-clip that keeps all box upper-whiskers visible.
    max_whisker = 1.0
    for v in slowdowns.values():
        arr = v
        q1, q3 = float(np.percentile(arr, 25)), float(np.percentile(arr, 75))
        whisker_hi = min(float(arr.max()), q3 + 1.5 * (q3 - q1))
        max_whisker = max(max_whisker, whisker_hi)
    boxplot_y_clip = max(1.2, round(max_whisker * 1.4, 1))

    # plot_slowdown_boxplot(slowdowns, system, base_path.replace(".png", f"_cap{'NO' if cap_ignored else CAP}_boxplot.png"), y_clip=boxplot_y_clip)
    plot_slowdown_violinplot(
        slowdowns, system,
        base_path.replace(".png", f"_cap{'NO' if cap_ignored else CAP}_violinplot.png"),
        y_clip=boxplot_y_clip,
        threshold=-999,
        min_cluster_frac=0.15,
    )
    print()
    print()